## imports

In [1]:
import os
import warnings
import logging
from typing import Tuple, Dict
from pypdf import PdfReader
from transformers import pipeline, AutoTokenizer, BitsAndBytesConfig
from accelerate import infer_auto_device_map
from transformers import AutoModelForCausalLM
import torch
import gc

# Suppress pypdf warnings about wrong pointing objects
logging.getLogger('pypdf').setLevel(logging.ERROR)

# model setup 

In [2]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# Verify quantization was applied
print(f"\nModel loaded successfully!")
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    #max_new_tokens=1024,
    batch_size=1,  # Process one at a time to reduce memory
    return_full_text=False,
    do_sample=True,  # Enable sampling instead of greedy decoding
    temperature=0.7,  # Control randomness (0.7 is good balance)
    top_p=0.9,  # Nucleus sampling
    repetition_penalty=1.2,  # Prevent repetitive output
    device_map="auto"
)


Loading model with 4-bit quantization...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/home/viper/Documents/GitHub/COMP5700-Project/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Model loaded successfully!
GPU Memory allocated: 0.90 GB
GPU Memory reserved: 0.94 GB


Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# I/O functions

## read two documents function

In [3]:
def read_document(pdf_path: str) -> str:
    """
    Load and validate a PDF document.
    Args:
        pdf_path: Path to the PDF document
    Returns:
        The text content from the PDF
    Raises:
        FileNotFoundError: If the PDF file does not exist
        ValueError: If the file is not a valid PDF or cannot be read
    """
    # Validate that the file exists
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    # Validate file extension
    if not pdf_path.lower().endswith('.pdf'):
        raise ValueError(f"File is not a PDF: {pdf_path}")

    # Extract text from PDF
    try:
        reader = PdfReader(pdf_path)  # instance
        text = ""
        for page in reader.pages:    
            text += page.extract_text()
        if not text.strip():
            raise ValueError(f"PDF contains no extractable text: {pdf_path}")

    except Exception as e:
        raise ValueError(f"Failed to read first PDF {pdf_path}: {str(e)}")

    return text

## print LLM output function

In [4]:
def PrintPrompt(prompt, promptType: str, LLMoutput):
    with open('LLMoutputs.txt', 'a', encoding='utf-8') as f:
        f.write(model.config._name_or_path) #model name
        f.write(f"\n...\n")
        f.write(prompt) # prompt used
        f.write(f"\n...\n")
        f.write(promptType) # prompt type
        f.write(f"\n...\n")
        f.write(str(LLMoutput)) # LLM output
        f.write(f"\n...\n")
        f.flush()  # Ensure data is written immediately
        print(f"print successfull")

# prompt functions

## zero shot

In [5]:
def zero_shot_prompt(doc: str,  max_chars: int = 120000) -> str:
    """
    Construct a zero-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
        max_chars: Maximum characters per document to avoid memory issues (default: 2000)
    
    Returns:
        A string containing the zero-shot prompt for the LLM
    """
    # Truncate documents significantly to avoid OOM errors
    doc_truncated = doc[:max_chars] + ("..." if len(doc) > max_chars else "")
    prompt = f"""You are an expert document analyst. Your task is to analyze two documents and identify key data elements present in each.

Document 1:
{doc_truncated}

Please analyze both documents and identify:
1. Key data elements (e.g., version numbers, dates, entity names, specifications)
2. Common elements that appear in both documents
3. Elements that are unique to each document
4. Any significant differences in how similar information is presented

Provide your analysis in a structured format."""
    
    return prompt

## Few Shot

In [6]:
def few_shot_prompt(doc: str, max_chars: int = 120000) -> str:
    """
    Construct a few-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
        max_chars: Maximum characters per document to avoid memory issues
    
    Returns:
        A string containing the few-shot prompt for the LLM
    """

    # Truncate documents to avoid OOM
    doc_truncated = doc[:max_chars] + ("..." if len(doc) > max_chars else "")

    # Few-shot examples
    example_1 = """
Example 1:

Document 1:
Version: 1.0
Date: 2023-01-01
Company: ABC Corp

Document 2:
Version: 1.1
Date: 2023-02-01
Company: ABC Corp

Output:
- Key Data Elements:
  - Version Numbers: 1.0, 1.1
  - Dates: 2023-01-01, 2023-02-01
  - Entity Name: ABC Corp
- Common Elements:
  - Company: ABC Corp
- Unique Elements:
  - Document 1: Version 1.0, Date 2023-01-01
  - Document 2: Version 1.1, Date 2023-02-01
- Differences:
  - Version and date updated in Document 2
"""

    example_2 = """
Example 2:

Document 1:
Product: Router X
IP: 192.168.1.1
Protocol: HTTP

Document 2:
Product: Router X
IP: 192.168.1.2
Protocol: HTTPS

Output:
- Key Data Elements:
  - Product Name: Router X
  - IP Addresses: 192.168.1.1, 192.168.1.2
  - Protocols: HTTP, HTTPS
- Common Elements:
  - Product: Router X
- Unique Elements:
  - Document 1: IP 192.168.1.1, HTTP
  - Document 2: IP 192.168.1.2, HTTPS
- Differences:
  - IP address and protocol differ
"""

    # Final prompt
    prompt = f"""You are an expert document analyst. Your task is to analyze the document and identify key data elements.

Here are some examples:

{example_1}

{example_2}

Now analyze the following:


{doc_truncated}


Please provide:
1. Key data elements
2. Common elements
3. Unique elements
4. Differences

Return the answer in a structured format similar to the examples above.
"""

    return prompt

## Chain of Thought

In [7]:
def chain_of_thought_prompt(doc: str, max_chars: int = 80000) -> str:
    """
    Construct a chain-of-thought prompt to identify key data elements in a document.

    Args:
        doc: Text content of the document
        max_chars: Maximum characters per document to avoid memory issues

    Returns:
        A string containing the chain-of-thought prompt for the LLM
    """

    # Truncate document to avoid OOM
    doc_truncated = doc[:max_chars] + ("..." if len(doc) > max_chars else "")

    prompt = f"""You are an expert document analyst. Your task is to analyze the document and identify key data elements.

Think step by step to:
1. Extract all relevant data elements (e.g., version numbers, dates, entity names, specifications)
2. Group similar elements together
3. Identify relationships or patterns among elements
4. Determine which elements are repeated or unique
5. Note any inconsistencies or differences in how information is presented

Document:
{doc_truncated}

After reasoning through the document, provide your final answer in the following structured format:

- Key Data Elements:
- Common Elements:
- Unique Elements:
- Differences:
"""

    return prompt

# prompt use

In [8]:
# Define the nine input combinations
files = [
    "cis-r1.pdf",
    "cis-r2.pdf",
    "cis-r3.pdf",
    "cis-r4.pdf",
]

OutputList = [[],[],[]] #zero, few, chain of thought

pdf_dir = "PDFS"

# Test each combination
for i, pdf in enumerate(files, 1):
#load the two docs
    print(f"\nInput-{i}: {pdf}")
    pdf_path = os.path.join(pdf_dir, pdf)
    doc = read_document(pdf_path)
    print(f"Successfully loaded document {pdf}")
    
#zero shot
    print(f"running zero shot")
    prompt = zero_shot_prompt(doc) # DO THIS SEQUENCALLY
    promptoutput = pipe(prompt)[0]['generated_text'] # make sure to use this! (outputs are actually a dictionary and its silly, but this gets around that)
    print("prompt generated, saving to LLMoutputs.txt")
    OutputList[0].append(promptoutput) # used for KDE
    torch.cuda.empty_cache() #clean up memory (we may need it!)
    gc.collect()
    
#fewshot
    print(f"running few shot")
    prompt = few_shot_prompt(doc)
    promptoutput = pipe(prompt)[0]['generated_text']
    print("Prompt generated, saving to LLMoutputs.txt")
    OutputList[1].append(promptoutput) # used for KDE
    torch.cuda.empty_cache() #clean up memory (we may need it!)
    gc.collect()
    PrintPrompt(prompt, "Few Shot", promptoutput)

# chain of thought
    print("running chain of thought")
    prompt = chain_of_thought_prompt(doc)
    promptoutput = pipe(prompt)[0]['generated_text']
    print("Prompt generated, saving to LLMoutputs.txt")
    OutputList[2].append(promptoutput)
    torch.cuda.empty_cache()
    gc.collect()
    PrintPrompt(prompt, "Chain of Thought", promptoutput)
print(f"\nAll outputs saved to LLMoutputs.txt")
        


Input-1: cis-r1.pdf


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Successfully loaded document cis-r1.pdf
running zero shot


/home/viper/Documents/GitHub/COMP5700-Project/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


prompt generated, saving to LLMoutputs.txt
running few shot


...

prompt generated, saving to LLMoutputs.txt
running few shot


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt generated, saving to LLMoutputs.txt


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


print successfull
running chain of thought
Prompt generated, saving to LLMoutputs.txt
print successfull

All outputs saved to LLMoutputs.txt


# key data elements

In [ ]:
def KDE(prompt: str) -> str:
    """
    Construct a zero-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
        max_chars: Maximum characters per document to avoid memory issues (default: 2000)
    
    Returns:
        A string containing the zero-shot prompt for the LLM
    """
    # Truncate documents significantly to avoid OOM errors (kept incase needed later)
    # doc1_truncated = doc1[:max_chars] + ("..." if len(doc1) > max_chars else "")
    # doc2_truncated = doc2[:max_chars] + ("..." if len(doc2) > max_chars else "")
    
    prompt = f"""You are an expert document analyst. Your task is to analyze the prompt given identify key data elements present in it.
prompt:
{prompt}

Please analyze the prompt given and generate a output in yaml format like so:
element1:
  name:
  requirements: 
    req1
    req2 
    req3 
"""
    
    return prompt

In [10]:
KDEFolder = "KDEs"

for PromptType, outputs in enumerate(OutputList):
    print(str(PromptType) + str(outputs))
    for item, output in enumerate(outputs):
        match PromptType:
            case 0: # zero shot
                KDEFilePath = "zero"
            case 1: # few shot
                KDEFilePath = "few"
            case 2: # chain of thought
                KDEFilePath = "chain"
        KDEFilePath = os.path.join(KDEFolder, files[item].replace(".pdf", "") + " "+ KDEFilePath + " KDEs.yaml")
        KDEoutput = pipe(KDE(output))[0]['generated_text']
        with open(KDEFilePath, 'w', encoding='utf-8') as f:
            f.write(str(KDEoutput))
            f.flush()

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0['\n```python\ndef analyze_documents(document1, document2):\n    analysis = {}\n    # Find common elements\n    common_elements = []\n    for element in document1.split():\n        if element in document2:\n            common_elements.append(element)\n    analysis["common_elements"] = common_elements\n\n    # Analyze differences\n    differences = []\n    for element in document1.split():\n        if element in document2:\n            differences.append((element, document1)) # Mark the difference\n    analysis["differences"] = differences\n\n    return analysis\n```\nHere\'s the analysis based on the provided python code. I\'ll explain why it works and what insights it provides.\n\n**Analysis:**\n\nThe `analyze_documents` function performs a comparative analysis of two text-based documents (`document1`, `document2`) to highlight similarities and differences. Let\'s dissect the approach and results:\n\n**1. Key Data Elements (Common Elements)**\n\n*   The core logic iterates through *e

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1['', '```\nPage 82 \nInternal Only - General \nCIS Controls: \nControls \nVersion Control IG 1 IG 2 IG 3 \nv8 \n4.8 Uninstall or Disable Unnecessary Services on \nEnterprise Assets and Software \n Uninstall or disable unnecessary services on enterprise assets and software, \nsuch as an unused file sharing service, web application module, or service \nfunction. \n   ● ● ●\nV7 \n14.7 Enforce Access Control to Data Through Automated \nTools \n Use an automated tool, such as host-based Data Loss Prevention, to enforce \naccess controls to data even when data is copied off a system. \n   ● ● ●\nV7 \n14.7 Enforce Access Control to Data Through Automatic \nImpersonate Permissions \nUse an automated impersonate permission, such as a Kubernetes Engine \nService account, to grant access to a specific topic. \n   ● ● ●\nV7 \n14.7 Automate Escalation with Role Based Access \nPolicies \nImplement role-based access policies to automate escalation of privileges \nwhen triggered. \n   ● ● ●\nV7', '--

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2['- Overall Assessment:\n```\n**- Key Data Elements:**\n*  Version Numbers: (e.g., v8.2, v7.6)\n*  Automation Levels: (Automated, Manual)\n*  Configuration Profiles: (e.g., CIS Controls)\n*  Specific Settings: (e.g., `--read-only-port`, `--eventRecordQPS`)\n*  Authentication Methods: (e.g., User Credentials, Certificates)\n- Recommended Practices: (e.g., Using TLS, Implementing audit logging)\n- Risk Considerations: (e.g., Exceeding resource limits, Maintaining outdated configurations)\n- System Requirements: (e.g., Specific Kubelet versions, Supported Operating Systems)\n\n- Common Elements:\n*  Security Policies: (Access control lists, network segmentation)\n*  Monitoring Tools: (Audit logs, Event Recording)\n*  Operating System Support: (Linux, Windows, etc.)\n*  Cloud Provider Integration: (AWS, Azure, Google Cloud)\n- Unique Elements:\n*  Customization Strategies: (Implementing custom security measures tailored to specific organizational needs)\n- Potential Issues: (Resource exha

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
